# Integrating the KA1 mobility dataset with Cost of Living data (Eurostat HICP)

This notebook:
1. Loads `df_mobility` from `dataframe_1000_examples.csv`
2. Loads `df_hicp` from `prc_hicp_aind_linear_csv.gz` (Eurostat `prc_hicp_aind` dataset)
3. Adds two new columns to `df_mobility`:
   - `Sending Country HICP` (right after `Sending Country`)
   - `Receiving Country HICP` (right after `Receiving Country`)
4. The join is done on the country name and on the **Academic Year**, which in the HICP dataset corresponds to `TIME_PERIOD`.

**Note on the chosen measure**: the Eurostat `prc_hicp_aind` file contains several metrics (`unit`) and hundreds of expenditure sub-categories (`coicop`). As a proxy for the overall "cost of living" per country/year, I used the measure most representative of the general cost of living:
- `unit = "Annual average index"` (annual average consumer price index)
- `coicop = "All-items HICP"` (full basket, not a single expenditure category)

If you had a different unit/coicop combination in mind, you can change the two filters in cell 4; the following cell prints the available options.

In [4]:
import pandas as pd
import numpy as np

MOBILITY_PATH = "dataframe_1000_examples.csv"
HICP_PATH = "prc_hicp_aind_linear.csv.gz"

## 1. Loading df_mobility

In [5]:
df_mobility = pd.read_csv(MOBILITY_PATH, index_col=0)
df_mobility.head()

,Academic Year,Mobility Duration,Field of Education,Participant Country,Education Level,Participant Gender,Fewer Opportunities,Participant Age,Sending Country,Sending City,Sending Organization,Receiving Country,Receiving City,Receiving Organization
167647,2016,121,Arts,France,ISCED-7 - Second cycle / Master’s or equivalen...,Male,Yes,25,France,METZ,Ecole Supérieure d'Art de Lorraine,Turkey,ISTANBUL,Sabanci University
21279,2019,42,Chemistry,Belgium,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,No,21,Belgium,Geel,THOMAS MORE KEMPEN VZW,Poland,Gdansk,UNIWERSYTET GDANSKI
483552,2017,139,Business and administration,France,ISCED-7 - Second cycle / Master’s or equivalen...,Male,No,24,France,BELFORT,ASSOCIATION POUR LA GESTION DE L'ECOLE SUPERIE...,Germany,BIETIGHEIM-BISSINGEN,MAGNA CAR TOP SYSTEMS GmbH
347842,2017,140,Journalism and reporting,United Kingdom,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,No,20,United Kingdom,SOUTHAMPTON,Southampton Solent University,Spain,VILLANUEVA DE GALLEGO ZARAGOZA,FUNDACION UNIVERSIDAD SAN JORGE
26018,2019,134,Economics,Czechia,ISCED-7 - Second cycle / Master’s or equivalen...,Female,No,23,Czechia,OSTRAVA PORUBA,VYSOKA SKOLA BANSKA - TECHNICKA UNIVERZITA OST...,Greece,PIRAEUS,UNIVERSITY OF PIRAEUS RESEARCH CENTER


## 2. Loading df_hicp

The gz file already contains a formatted csv (columns: `DATAFLOW, LAST UPDATE, freq, unit, coicop, geo, TIME_PERIOD, OBS_VALUE, OBS_FLAG, CONF_STATUS`).

In [22]:
df_hicp = pd.read_csv(HICP_PATH, compression="gzip", low_memory=False)
df_hicp.sample(20, random_state=42)

,DATAFLOW,LAST UPDATE,freq,unit,coicop,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
337210,ESTAT:PRC_HICP_AIND(1.0),06/02/26 23:00:00,Annual,Annual average rate of change,Butter,Malta,2020,-3.60,NaN,NaN
266951,ESTAT:PRC_HICP_AIND(1.0),06/02/26 23:00:00,Annual,Annual average index,Legal services and accountancy,Germany,2015,100.00,NaN,NaN
458626,ESTAT:PRC_HICP_AIND(1.0),06/02/26 23:00:00,Annual,Annual average rate of change,Accessories for personal transport equipment,Finland,2023,7.00,NaN,NaN
536366,ESTAT:PRC_HICP_AIND(1.0),06/02/26 23:00:00,Annual,Annual average rate of change,Accommodation services,Finland,2024,1.80,NaN,NaN
55972,ESTAT:PRC_HICP_AIND(1.0),06/02/26 23:00:00,Annual,Annual average index,Wine,France,2022,113.42,NaN,NaN
514950,ESTAT:PRC_HICP_AIND(1.0),06/02/26 23:00:00,Annual,Annual average rate of change,"Museums, libraries, zoological gardens",Germany,2021,2.60,NaN,NaN
308426,ESTAT:PRC_HICP_AIND(1.0),06/02/26 23:00:00,Annual,Annual average index,Overall index excluding energy and unprocessed...,"European Union (EU6-1958, EU9-1973, EU10-1981,...",2010,92.97,NaN,NaN
381804,ESTAT:PRC_HICP_AIND(1.0),06/02/26 23:00:00,Annual,Annual average rate of change,Footwear for infants and children,Bulgaria,2015,1.80,NaN,NaN
380831,ESTAT:PRC_HICP_AIND(1.0),06/02/26 23:00:00,Annual,Annual average rate of change,Footwear for men,Albania,2022,-0.60,d,NaN
283301,ESTAT:PRC_HICP_AIND(1.0),06/02/26 23:00:00,Annual,Annual average index,"Non-energy industrial goods, non-durables only",Sweden,2020,109.19,NaN,NaN


In [32]:
UNIT = "Annual average index"
COICOP = "All-items HICP"

df_hicp_filtered = df_hicp[(df_hicp["unit"] == UNIT) & (df_hicp["coicop"] == COICOP)].copy()
df_hicp_filtered["TIME_PERIOD"] = df_hicp_filtered["TIME_PERIOD"].astype(int)
df_hicp_filtered = df_hicp_filtered[["geo", "TIME_PERIOD", "OBS_VALUE"]].rename(columns={"OBS_VALUE": "HICP"})
print(df_hicp_filtered.shape)
df_hicp_filtered.head()

(1229, 3)


,geo,TIME_PERIOD,HICP
5006,Albania,2016,101.51
5007,Albania,2017,104.76
5008,Albania,2018,106.59
5009,Albania,2019,108.39
5010,Albania,2020,110.74


## 3. Country name normalization

The `geo` field in `df_hicp` uses the official Eurostat names, which in a few cases differ slightly from the ones used in `df_mobility`
(e.g. `Czech Republic` vs `Czechia`, `Turkey` vs `Türkiye`). We map these differences to maximize the number of matches.

Non-European countries present in `df_mobility` (e.g. Australia, Canada, China, South Korea, United Arab Emirates, Vietnam, Cambodia, Palestine, Ukraine) are not covered by the Eurostat HICP dataset: for these the resulting value will correctly be `NaN`.

**Note on the UK**: in the Eurostat file, the United Kingdom's time series stops in 2019 (following Brexit). Mobilities with `Sending/Receiving Country = "United Kingdom"` and `Academic Year >= 2020` will therefore legitimately have `NaN`, not because of a join error.

In [26]:
# map to align df_mobility country names with the ones used in df_hicp (the 'geo' column)
COUNTRY_NAME_MAP = {
    "Czech Republic": "Czechia",
    "Turkey": "Türkiye",
    "The Republic of North Macedonia": "North Macedonia",
}


def normalize_country_name(name):
    if pd.isna(name):
        return np.nan
    return COUNTRY_NAME_MAP.get(name, name)

## 4. Building the final dataframe with the join

In [27]:
df = df_mobility.copy()

df["_sending_country_name"] = df["Sending Country"].apply(normalize_country_name)
df["_receiving_country_name"] = df["Receiving Country"].apply(normalize_country_name)

# join for the sending country, on the academic year
hicp_sending = df_hicp_filtered.rename(
    columns={"geo": "_sending_country_name", "TIME_PERIOD": "Academic Year", "HICP": "Sending Country HICP"}
)
df = df.merge(hicp_sending, on=["_sending_country_name", "Academic Year"], how="left")

# join for the receiving country, on the academic year
hicp_receiving = df_hicp_filtered.rename(
    columns={"geo": "_receiving_country_name", "TIME_PERIOD": "Academic Year", "HICP": "Receiving Country HICP"}
)
df = df.merge(hicp_receiving, on=["_receiving_country_name", "Academic Year"], how="left")

# drop the helper columns used only for the join
df = df.drop(columns=["_sending_country_name", "_receiving_country_name"])

In [28]:
# reorder columns to insert the new features right after Sending Country / Receiving Country
new_column_order = []
for col in df_mobility.columns:
    new_column_order.append(col)
    if col == "Sending Country":
        new_column_order.append("Sending Country HICP")
    elif col == "Receiving Country":
        new_column_order.append("Receiving Country HICP")

df_mobility_hicp = df[new_column_order]
df_mobility_hicp.head()

,Academic Year,Mobility Duration,Field of Education,Participant Country,Education Level,Participant Gender,Fewer Opportunities,Participant Age,Sending Country,Sending Country HICP,Sending City,Sending Organization,Receiving Country,Receiving Country HICP,Receiving City,Receiving Organization
0,2016,121,Arts,France,ISCED-7 - Second cycle / Master’s or equivalen...,Male,Yes,25,France,100.31,METZ,Ecole Supérieure d'Art de Lorraine,Turkey,107.66,ISTANBUL,Sabanci University
1,2019,42,Chemistry,Belgium,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,No,21,Belgium,107.77,Geel,THOMAS MORE KEMPEN VZW,Poland,104.80,Gdansk,UNIWERSYTET GDANSKI
2,2017,139,Business and administration,France,ISCED-7 - Second cycle / Master’s or equivalen...,Male,No,24,France,101.47,BELFORT,ASSOCIATION POUR LA GESTION DE L'ECOLE SUPERIE...,Germany,102.10,BIETIGHEIM-BISSINGEN,MAGNA CAR TOP SYSTEMS GmbH
3,2017,140,Journalism and reporting,United Kingdom,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,No,20,United Kingdom,103.40,SOUTHAMPTON,Southampton Solent University,Spain,101.69,VILLANUEVA DE GALLEGO ZARAGOZA,FUNDACION UNIVERSIDAD SAN JORGE
4,2019,134,Economics,Czechia,ISCED-7 - Second cycle / Master’s or equivalen...,Female,No,23,Czechia,107.80,OSTRAVA PORUBA,VYSOKA SKOLA BANSKA - TECHNICKA UNIVERZITA OST...,Greece,102.46,PIRAEUS,UNIVERSITY OF PIRAEUS RESEARCH CENTER


## 5. Saving the result

In [33]:
df_mobility_hicp.to_csv("df_mobility_hicp.csv")
print("Saved df_mobility_hicp.csv")

Saved df_mobility_hicp.csv
